# Amaliy mashgʻulot 6. Diqqat mexanizmi (attention), pozitsion kodlash va BERT kontekstual embeddinglari

6-laboratoriyada olmosh gapdagi boshqa soʻzlarga qanday «qarashini», sinusoidalar pozitsiyani qanday kodlashini va koʻp maʼnoli soʻz vektori oʻz maʼnosi tomon qanday siljishini koʻrdingiz. Bu yerda buni kodda takrorlaymiz:

1. scaled dot-product attention ni `numpy` da qadamma-qadam hisoblaymiz: $QK^\top$, $\sqrt{d}$ ga boʻlish, satrlar boʻyicha softmax, $V$ ning vaznli yigʻindisi;
2. olmosh nimaga ishora qilishini tushunish uchun bitta diqqat qatlami nega yetmasligini va ikki qatlam buni qanday hal qilishini koʻramiz;
3. koʻp boshli diqqatni (tenzorlar oʻlchamlari) va niqoblarni tahlil qilamiz: sababiy (GPT dagidek) va toʻldirish (padding) niqobi;
4. sinusoidal pozitsion kodlashni quramiz va usiz diqqat soʻzlar tartibini koʻrmasligini tekshiramiz;
5. koʻp maʼnoli soʻzning kontekstual vektorlarini olamiz: avval oʻyinchoq modelda, soʻng (Colab da) haqiqiy koʻp tilli XLM-RoBERTa modelida, va u bilan «niqob ostidagi soʻzni top» oʻyinini oʻynaymiz.

*«NLP algoritmlari» kursining 6-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Scaled dot-product attention qadamma-qadam

$x_i$ vektorli har bir token uchta vektor oladi: **soʻrov** $q_i = x_i W_Q$ («men nimani qidiryapman»), **kalit** $k_i = x_i W_K$ («men nimani taklif qila olaman») va **qiymat** $v_i = x_i W_V$ («men nimani uzataman»). Unda

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V.$$

Vaznlar matritsasining $i$-satri $i$-token qaysi tokenlarga «qarashini» koʻrsatadi; har bir satr yigʻindisi 1 ga teng. $\sqrt{d}$ ga boʻlish skalyar koʻpaytmalar oʻlcham bilan oʻsib, softmax ni haddan tashqari keskin qilib yubormasligi uchun kerak.

Laboratoriyadagi misolni olamiz: «Mushuk devordan sakray olmadi, chunki u charchagan edi». Har bir soʻzda qoʻlda berilgan 4 ta belgi bor: *jonli*, *buyum*, *olmosh*, *belgi* (ega haqida biror narsa aytadigan kesim soʻz). Oʻzbek tilida «u» olmoshida jins yoʻq, shuning uchun model grammatikaga emas, maʼnoga tayanishi kerak. Yordamchi soʻzlar va feʼl bu yerda nolga teng: bu masala uchun ular muhim emas. $W$ matritsalari ham qoʻlda berilgan: olmosh «belgi qayerda?» deb soʻraydi, belgi javob beradi, qiymat esa *jonli* va *buyum* maʼno belgilarini uzatadi.


Endi mexanizmning oʻzi. «u» olmoshining satrini kuzatamiz.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
FEATS = ["jonli", "buyum", "olmosh", "belgi"]
LEX = {
    "mushuk":     [1, 0, 0, 0],
    "devordan":   [0, 1, 0, 0],
    "u":          [0, 0, 1, 0],
    "charchagan": [1, 0, 0, 1],   # jonli mavjudot haqidagi kesim
    "baland":     [0, 1, 0, 1],   # buyum haqidagi kesim
}

def embed(sentence):
    toks = sentence.split()
    X = np.array([LEX.get(t, [0, 0, 0, 0]) for t in toks], dtype=float)   # notanish soʻzlar — nol vektor
    return toks, X

S1 = "mushuk devordan sakray olmadi chunki u charchagan edi"
S2 = "mushuk devordan sakray olmadi chunki u baland edi"
toks, X = embed(S1)
print(pd.DataFrame(X, index=toks, columns=FEATS).astype(int).to_string())

d = 4
W_Q1 = np.zeros((4, d)); W_Q1[2, 0] = 3                    # olmosh soʻraydi: «belgi qayerda?»
W_K1 = np.zeros((4, d)); W_K1[3, 0] = 3                    # belgi javob beradi: «men shu yerdaman»
W_V1 = np.zeros((4, d)); W_V1[0, 0] = 1; W_V1[1, 1] = 1    # «jonli» va «buyum» ni uzatamiz

def softmax_rows(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)      # barqarorlik uchun maksimumni ayiramiz
    E = np.exp(Z)
    return E / E.sum(axis=-1, keepdims=True)

Q, K, V = X @ W_Q1, X @ W_K1, X @ W_V1
print("1-qadam. Q, K, V (n × d):", Q.shape, K.shape, V.shape)
print(pd.DataFrame(np.hstack([Q[:, :1], K[:, :1], V[:, :2]]), index=toks,
                   columns=["q[0]", "k[0]", "v[0]", "v[1]"]).to_string())

scores = Q @ K.T
scaled = scores / np.sqrt(d)
A = softmax_rows(scaled)
out = A @ V

i = toks.index([t for t in toks if t in ("u",)][0])
print(f"\n2–4-qadamlar «{toks[i]}» satri uchun:")
print("   q·k      :", scores[i])
print("   / √d     :", scaled[i])
print("   exp      :", np.exp(scaled[i]).round(2), " yigʻindi", np.exp(scaled[i]).sum().round(2))
print("   softmax  :", A[i].round(3))
print("\n5-qadam. Chiqish = A·V, satr", f"«{toks[i]}»:", ", ".join(f"{f} = {x:.3f}" for f, x in zip(FEATS[:2], out[i, :2])))
print("A satrlari yigʻindilari:", A.sum(axis=1).round(6))

Olmosh deyarli butun diqqatini (0,93) «charchagan» kesimiga berdi va undan *jonli* belgisini oldi. Qolgan tokenlarning soʻrovi nolga teng, shuning uchun ularning diqqati teng taqsimlangan (har bir tokenga $1/8$). Softmax ni `scipy` bilan tekshiramiz va vaznlar matritsasini chizamiz.

In [ ]:
from scipy.special import softmax as sp_softmax

print("softmax scipy bilan mos keladi:", np.allclose(A, sp_softmax(scaled, axis=1)))

def show_attention(ax, A, labels, title):
    ax.imshow(A, cmap="Blues", vmin=0, vmax=1)
    for r in range(A.shape[0]):
        for c in range(A.shape[1]):
            ax.text(c, r, f"{A[r, c]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if A[r, c] > 0.6 else "black")
    ax.set_xticks(range(len(labels)), labels, rotation=60, fontsize=8)
    ax.set_yticks(range(len(labels)), labels, fontsize=8)
    ax.set_xlabel("kimga qaraymiz (kalitlar)")
    ax.set_ylabel("kim qaraydi (soʻrovlar)")
    ax.set_title(title)

fig, ax = plt.subplots(figsize=(6, 5))
show_attention(ax, A, toks, "Diqqat vaznlari, bitta qatlam")
plt.tight_layout()
plt.show()

### PyTorch bilan tekshirish (faqat Colab da)

PyTorch da tayyor `torch.nn.functional.scaled_dot_product_attention` funksiyasi bor: u aynan shu formulani hisoblaydi (odatda $\sqrt{d}$ ga boʻladi, bu yerda $d$ — soʻrov uzunligi). Quyidagi katak uni bizning `numpy` bilan solishtiradi.

In [ ]:
try:
    import torch
    import torch.nn.functional as F
    tq, tk, tv = (torch.tensor(M, dtype=torch.float32)[None] for M in (Q, K, V))   # paket oʻqini qoʻshdik: (1, n, d)
    out_torch = F.scaled_dot_product_attention(tq, tk, tv)[0].numpy()
    print("PyTorch:", out_torch.shape, "numpy bilan mos keladi:", np.allclose(out_torch, out, atol=1e-5))
except Exception as e:
    print("PyTorch mavjud emas:", e)

## 2. Nega bitta qatlam yetmaydi: «u» nimaga ishora qiladi?

Ikki gapni solishtiramiz: «…chunki **u charchagan** edi» (gap mushuk haqida) va «…chunki **u baland** edi» (gap devor haqida). Bitta qatlamda olmosh soʻrovi $q = x W_Q$ faqat olmoshning oʻz vektoriga bogʻliq, «mushuk» va «devordan» kalitlari esa boshqa soʻzlarga bogʻliq emas. Demak, olmosh satridagi «mushuk : devordan» vaznlari nisbati ikkala gapda bir xil. Ikkinchi qatlam kerak:

* **1-qatlam**: olmosh kesimga qaraydi va undan maʼno belgisini oladi («charchagan» da *jonli*, «baland» da *buyum*);
* **2-qatlam**: boyitilgan olmosh shu belgili otni qidiradi.

Qatlamlar orasida **qoldiq bogʻlanish** (residual) ishlaydi: $h = x + \text{Attention}(x)$, shuning uchun tokenning dastlabki belgilari yoʻqolmaydi. Haqiqiy transformerlarda 6–24 ta qatlam bor va diqqatning bunday «zanjirlari» oʻqitish jarayonida oʻz-oʻzidan shakllanadi.

Quyidagi natijada «mushuk : devordan» vaznlari nisbatini solishtiring: 1-qatlamda u ikkala gapda 1,00 ga teng, 2-qatlamda esa olmosh birinchi gapda «mushuk» ni, ikkinchisida «devordan» ni ishonch bilan tanlaydi.

In [ ]:
def attention(X, W_Q, W_K, W_V, mask=None):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    S = Q @ K.T / np.sqrt(Q.shape[-1])
    if mask is not None:
        S = np.where(mask, S, -1e9)            # taqiqlangan pozitsiyalar −∞ oladi
    A = softmax_rows(S)
    return A @ V, A

# 2-qatlam: soʻrov — maʼno belgilari, kalit — otning maʼno belgilari (belgi va olmosh emas)
W_Q2 = np.zeros((4, d)); W_Q2[0, 0] = 3; W_Q2[1, 1] = 3
W_K2 = np.zeros((4, d)); W_K2[0, 0] = 3; W_K2[1, 1] = 3
W_K2[2, :2] = -3; W_K2[3, :2] = -3
W_V2 = np.eye(4)

def two_layers(X):
    out1, A1 = attention(X, W_Q1, W_K1, W_V1)
    H1 = X + out1                               # qoldiq bogʻlanish
    out2, A2 = attention(H1, W_Q2, W_K2, W_V2)
    return H1 + out2, A1, A2

ANIM, OBJ = "mushuk", "devordan"
results = {}
for s in (S1, S2):
    tk_, X_ = embed(s)
    H2, A1, A2 = two_layers(X_)
    p = [j for j, t in enumerate(tk_) if t in ("u",)][0]
    a, o = tk_.index(ANIM), tk_.index(OBJ)
    results[s] = (tk_, A1[p], A2[p])
    print(s)
    print(f"   1-qatlam: «{tk_[p]}» → " + ", ".join(f"{t} {w:.2f}" for t, w in zip(tk_, A1[p]) if w > 0.05),
          f"|  {ANIM}:{OBJ} = {A1[p, a] / A1[p, o]:.2f}")
    print(f"   2-qatlam: «{tk_[p]}» → " + ", ".join(f"{t} {w:.2f}" for t, w in zip(tk_, A2[p]) if w > 0.05),
          f"|  {ANIM}:{OBJ} = {A2[p, a] / A2[p, o]:.2f}")

fig, axes = plt.subplots(2, 2, figsize=(11, 5), sharey=True)
for r, s in enumerate((S1, S2)):
    tk_, a1, a2 = results[s]
    for c, (w, name) in enumerate([(a1, "1-qatlam"), (a2, "2-qatlam")]):
        ax = axes[r, c]
        cols = ["#D9532C" if t in (ANIM, OBJ) else "#2B55C9" for t in tk_]
        ax.bar(range(len(tk_)), w, color=cols)
        ax.set_xticks(range(len(tk_)), tk_, rotation=30, fontsize=8)
        ax.set_ylim(0, 1)
        pred = [t for t in tk_ if LEX.get(t, [0, 0, 0, 0])[3] == 1][0]      # shu gapning belgi-kesimi
        ax.set_title(f"{name}: …{pred}", fontsize=10)
fig.suptitle("Olmosh qayerga qaraydi")
plt.tight_layout()
plt.show()

## 3. Koʻp boshli diqqat va niqoblar

**Koʻp boshli diqqat** (multi-head): $d_{model}$ oʻlchamli bitta diqqat oʻrniga $d_k = d_{model}/h$ oʻlchamli $h$ ta «bosh» hisoblanadi, har birining oʻz $W_Q, W_K, W_V$ matritsalari bor; natijalar yopishtiriladi va $W_O$ ga koʻpaytiriladi. Turli boshlar turli munosabatlarni kuzatishi mumkin (qoʻshni soʻz, ega, olmosh…). Tasodifiy maʼlumotlarda tenzorlar oʻlchamlarini kuzatamiz.

In [ ]:
rng = np.random.default_rng(0)
n, d_model, h = 8, 12, 3
d_k = d_model // h
Xr = rng.normal(size=(n, d_model))
W_Q, W_K, W_V, W_O = (rng.normal(size=(d_model, d_model)) / np.sqrt(d_model) for _ in range(4))

Q = Xr @ W_Q
K = Xr @ W_K
V = Xr @ W_V
print("X              ", Xr.shape)
print("Q = X·W_Q      ", Q.shape)
Qh = Q.reshape(n, h, d_k).transpose(1, 0, 2)      # (bosh, token, d_k)
Kh = K.reshape(n, h, d_k).transpose(1, 0, 2)
Vh = V.reshape(n, h, d_k).transpose(1, 0, 2)
print("Q boshlar boʻyicha", Qh.shape)
S = Qh @ Kh.transpose(0, 2, 1) / np.sqrt(d_k)
print("ballar          ", S.shape)
A_heads = softmax_rows(S)
heads = A_heads @ Vh
print("boshlar chiqishi ", heads.shape)
concat = heads.transpose(1, 0, 2).reshape(n, d_model)
print("yopishtirish     ", concat.shape)
Y = concat @ W_O
print("chiqish = ·W_O   ", Y.shape)

D, H = 768, 12
print(f"\nBERT-base: d_model={D}, h={H}, d_k={D // H}; bitta diqqat qatlamidagi parametrlar: 4·{D}² + 4·{D} = {4 * D * D + 4 * D:,}")

fig, axes = plt.subplots(1, h, figsize=(11, 3.4))
for k in range(h):
    axes[k].imshow(A_heads[k], cmap="Blues", vmin=0, vmax=1)
    axes[k].set_title(f"bosh {k + 1}")
    axes[k].set_xlabel("kalit")
axes[0].set_ylabel("soʻrov")
plt.tight_layout()
plt.show()

**Niqoblar.** BERT gapni toʻliq koʻradi (ikki tomonlama diqqat). Generativ model (GPT) $i$-soʻzni bashorat qilganda kelajakdagi soʻzlarga qaramasligi kerak, shuning uchun ballarga **sababiy niqob** qoʻshiladi: diagonaldan yuqorida $-\infty$, softmax dan keyin bu vaznlar 0 ga teng. **Toʻldirish niqobi** (padding mask) turli uzunlikdagi gaplar bitta paketga (batch) yigʻilganda kerak: `[PAD]` tokenlariga qarash mumkin emas.

In [ ]:
toks1, X1 = embed(S1)
n1 = len(toks1)
rng = np.random.default_rng(1)
Wq, Wk = rng.normal(size=(4, 4)), rng.normal(size=(4, 4))
X1n = X1 + rng.normal(scale=0.3, size=X1.shape)            # vaznlar bir xil boʻlmasligi uchun biroz shovqin

causal = np.tril(np.ones((n1, n1), dtype=bool))            # True — qarash mumkin
_, A_full = attention(X1n, Wq, Wk, np.eye(4))
_, A_causal = attention(X1n, Wq, Wk, np.eye(4), mask=causal)
print("sababiy niqob (1 — qarash mumkin):")
print(causal.astype(int))
print("3-satr niqobsiz: ", A_full[2].round(2))
print("3-satr niqob bilan:", A_causal[2].round(2))

# toʻldirish niqobi: qisqa gap n1 uzunlikkacha [PAD] bilan toʻldirilgan
short = "mushuk charchagan edi".split()
toks2 = short + ["[PAD]"] * (n1 - len(short))
X2 = np.vstack([embed("mushuk charchagan edi")[1], np.zeros((n1 - len(short), 4))]) + rng.normal(scale=0.3, size=(n1, 4))
pad_mask = np.array([t != "[PAD]" for t in toks2])[None, :].repeat(n1, axis=0)
_, A_pad = attention(X2, Wq, Wk, np.eye(4), mask=pad_mask)
print("\n", toks2)
print("1-satr toʻldirish niqobi bilan:", A_pad[0].round(2), "yigʻindi", A_pad[0].sum().round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
show_attention(axes[0], A_full, toks1, "Niqobsiz (BERT)")
show_attention(axes[1], A_causal, toks1, "Sababiy niqob (GPT)")
plt.tight_layout()
plt.show()

## 4. Pozitsion kodlash

Diqqat — bu vaznli yigʻindi, tokenlar qaysi tartibda turgani unga baribir. Soʻzlar oʻrni almashtirilsa, har bir token oʻsha chiqish vektorini oladi (u bilan birga oʻrin almashgan holda). Tartib ahamiyatli boʻlishi uchun token vektoriga pozitsiya vektori qoʻshiladi. Dastlabki transformerda u sinusoidal:

$$PE_{(p,\,2i)}=\sin\frac{p}{10000^{2i/d}},\qquad PE_{(p,\,2i+1)}=\cos\frac{p}{10000^{2i/d}}.$$

Kichik koordinatalar tez, kattalari sekin tebranadi — xuddi soat millari kabi. $PE_p\cdot PE_{p+k}=\sum_i\cos(k\,\omega_i)$ skalyar koʻpaytmasi $p$ pozitsiyaning oʻziga emas, faqat $k$ masofaga bogʻliq. (BERT formula oʻrniga pozitsiya vektorlarini oʻrganadi, lekin gʻoya bir xil.)

2-boʻlimdagi ikki qatlamli modelda tekshiramiz: gapni teskari tartibda oʻqiymiz. Soʻng pozitsion vektorlar xaritasini chizamiz va $PE_p\cdot PE_{p+k}$ faqat $k$ ga bogʻliqligini tekshiramiz: $p=5$ va $p=20$ uchun egri chiziqlar ustma-ust tushadi.

In [ ]:
def positional_encoding(n_pos, d):
    pos = np.arange(n_pos)[:, None]
    i = np.arange(0, d, 2)[None, :]
    angle = pos / 10000 ** (i / d)
    PE = np.zeros((n_pos, d))
    PE[:, 0::2] = np.sin(angle)
    PE[:, 1::2] = np.cos(angle)
    return PE

toks1, X1 = embed(S1)
order = list(range(len(toks1)))[::-1]                       # teskari tartibda oʻqiymiz
p = [j for j, t in enumerate(toks1) if t in ("u",)][0]
p_rev = order.index(p)
print("asl tartib:  ", " ".join(toks1))
print("teskari tartib:", " ".join(toks1[j] for j in order))

H_orig, _, _ = two_layers(X1)
H_rev, _, _ = two_layers(X1[order])
print(f"\npozitsiyalarsiz: «{toks1[p]}» chiqishi bir xilmi?", np.allclose(H_orig[p], H_rev[p_rev]),
      "  max|Δ| =", np.abs(H_orig[p] - H_rev[p_rev]).max().round(4))

PE = positional_encoding(len(toks1), 4)
H_orig_pe, _, _ = two_layers(X1 + PE)
H_rev_pe, _, _ = two_layers(X1[order] + PE)                 # pozitsiyalar joyida, soʻzlar oʻrni almashgan
print(f"pozitsiyalar bilan: «{toks1[p]}» chiqishi bir xilmi?", np.allclose(H_orig_pe[p], H_rev_pe[p_rev]),
      "  max|Δ| =", np.abs(H_orig_pe[p] - H_rev_pe[p_rev]).max().round(4))
print("\nPE birinchi 3 pozitsiya uchun (d=4):")
print(PE[:3].round(3))

PE64 = positional_encoding(50, 64)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
im = axes[0].imshow(PE64, cmap="RdBu", aspect="auto", vmin=-1, vmax=1)
axes[0].set_xlabel("koordinata")
axes[0].set_ylabel("pozitsiya")
axes[0].set_title("Pozitsion vektorlar, d = 64")
plt.colorbar(im, ax=axes[0])
for j in (0, 4, 10, 30):
    axes[1].plot(PE64[:, j], label=f"koord. {j}")
axes[1].set_xlabel("pozitsiya")
axes[1].legend(fontsize=7)
axes[1].set_title("Tez va sekin toʻlqinlar")
k = np.arange(0, 25)
for p0, style in ((5, "-"), (20, "--")):
    axes[2].plot(k, [PE64[p0] @ PE64[p0 + kk] for kk in k], style, label=f"p = {p0}")
axes[2].set_xlabel("masofa k")
axes[2].set_ylabel("PE_p · PE_{p+k}")
axes[2].legend()
axes[2].set_title("Faqat k ga bogʻliq")
plt.tight_layout()
plt.show()
print("egri chiziqlar orasidagi maksimal farq:", np.abs([PE64[5] @ PE64[5 + kk] - PE64[20] @ PE64[20 + kk] for kk in k]).max().round(10))

## 5. Statik va kontekstual embeddinglar

Word2Vec da (5-amaliy mashgʻulot) «koʻk» soʻzining barcha maʼnolari uchun bitta vektor bor: rang, koʻkat va osmon bir nuqtada aralashgan. Transformerda qatlam chiqishidagi soʻz vektori — kontekst soʻzlari vektorlarining diqqat vaznlari bilan aralashmasi, shuning uchun u har bir gapda oʻziga xos.

Oʻyinchoq: kontekst soʻzlarining statik 2 oʻlchamli vektorlari bor (qoʻlda berilgan, uchta maʼno guruhi), «koʻk» guruhlar orasida turadi. $Q=K=V=X$ boʻlgan bitta diqqat qadami kontekstual vektor beradi: $c=\sum_j \alpha_j x_j$.

In [ ]:
import re

static = {
    # rang
    "koʻylak": (0.2, 2.0), "kiyib": (-0.3, 1.8), "rangga": (0.1, 2.2), "boʻyashdi": (-0.2, 1.7), "devorni": (0.4, 1.6),
    # koʻkat
    "bozordan": (-1.8, -0.9), "bogʻ": (-1.6, -1.2), "somsaga": (-1.9, -1.0), "qoʻshdi": (-1.4, -0.8),
    # osmon
    "yulduzlar": (1.8, -1.0), "charaqlardi": (1.5, -0.8), "qushlar": (1.7, -1.2), "parvoz": (1.6, -0.9),
}
TARGET = "koʻk"
static_target = (0.2, 0.5)       # guruhlar orasida, eng koʻp uchraydigan «rang» maʼnosiga yaqinroq
toy_sentences = [
    ("U koʻk koʻylak kiyib keldi", "rang"),
    ("Devorni koʻk rangga boʻyashdi", "rang"),
    ("Bozordan bir bogʻ koʻk olib keldim", "koʻkat"),
    ("Somsaga koʻk qoʻshdi", "koʻkat"),
    ("Koʻkda yulduzlar charaqlardi", "osmon"),
    ("Qushlar koʻkka parvoz qildi", "osmon"),
]
GROUPS = {"rang": ["koʻylak", "kiyib", "rangga", "boʻyashdi", "devorni"],
          "koʻkat": ["bozordan", "bogʻ", "somsaga", "qoʻshdi"],
          "osmon": ["yulduzlar", "charaqlardi", "qushlar", "parvoz"]}

def words(s):
    return re.findall(r"\w+", s.lower())

def contextual(sentence):
    ws = [w for w in words(sentence) if w.startswith(TARGET) or w in static]
    Xs = np.array([static_target if w.startswith(TARGET) else static[w] for w in ws])
    t = [j for j, w in enumerate(ws) if w.startswith(TARGET)][0]
    alpha = softmax_rows(Xs[t] @ Xs.T / np.sqrt(2))          # Q = K = V = X
    return alpha @ Xs, ws, alpha

ctx = []
for s, sense in toy_sentences:
    c, ws, alpha = contextual(s)
    ctx.append(c)
    print(f"{s:<36} [{sense}]  α: " + ", ".join(f"{w} {a:.2f}" for w, a in zip(ws, alpha)), f"→ c = {c.round(2)}")
ctx = np.array(ctx)
print("«" + TARGET + "» ning statik vektori hamma gapda bitta:", static_target, "→ istalgan ikki qoʻllanish orasidagi kosinus = 1.00")

Cn = ctx / np.linalg.norm(ctx, axis=1, keepdims=True)
labels = [f"{i + 1}:{sense}" for i, (_, sense) in enumerate(toy_sentences)]
print("\nkontekstual vektorlar kosinuslari:")
print(pd.DataFrame(Cn @ Cn.T, index=labels, columns=labels).round(2).to_string())

fig, ax = plt.subplots(figsize=(6.5, 5.5))
colors = dict(zip(GROUPS, ["#2B55C9", "#17907E", "#D9532C"]))
for g, ws in GROUPS.items():
    pts = np.array([static[w] for w in ws])
    ax.scatter(pts[:, 0], pts[:, 1], color=colors[g], alpha=0.35, label=g)
    for w, (x, y) in zip(ws, pts):
        ax.annotate(w, (x, y), fontsize=7, color="gray", xytext=(3, 3), textcoords="offset points")
ax.scatter(*static_target, marker="*", s=250, color="black", label=TARGET + " (statik)")
for c, (_, sense) in zip(ctx, toy_sentences):
    ax.annotate("", xy=c, xytext=static_target, arrowprops=dict(arrowstyle="->", color=colors[sense]))
    ax.scatter(*c, marker="*", s=120, color=colors[sense])
ax.legend(fontsize=8, loc="lower center")
ax.set_title("«" + TARGET + "» kontekstual vektorlari oʻz maʼnosi tomon siljiydi")
plt.tight_layout()
plt.show()

Statik vektor oltita qoʻllanishning hammasida bitta (kosinus 1,00), kontekstual vektorlar esa bir maʼnoda bir-biriga yaqin va boshqa maʼnolar vektorlaridan uzoq.

### Haqiqiy modelda (faqat Colab da)

**XLM-RoBERTa base** (`FacebookAI/xlm-roberta-base`) ni olamiz: koʻp tilli BERT ga oʻxshash kodlovchi, 12 qatlam, taxminan 280 mln parametr, 100 tildagi matnlarda oʻqitilgan, ular orasida oʻzbek va rus tillari bor. Yuklab olish hajmi taxminan 1,1 GB (Colab da 1–2 daqiqa). Agar bu uzoq boʻlsa, `MODEL_ID` ni `distilbert/distilbert-base-multilingual-cased` ga almashtiring (taxminan 540 MB, 6 qatlam). `transformers` va `torch` kutubxonalari Colab da allaqachon oʻrnatilgan.

In [ ]:
MODEL_ID = "FacebookAI/xlm-roberta-base"     # ~1,1 GB; zaxira variant: "distilbert/distilbert-base-multilingual-cased" (~540 MB)
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForMaskedLM
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mlm = AutoModelForMaskedLM.from_pretrained(MODEL_ID)
    mlm.eval()
    n_params = sum(p.numel() for p in mlm.parameters())
    print(f"{MODEL_ID}: {mlm.config.num_hidden_layers} qatlam, hidden = {mlm.config.hidden_size}, {n_params / 1e6:.0f} mln parametr, niqob = {tok.mask_token}")
    for s, _ in toy_sentences[:2]:
        print("  ", s, "→", tok.tokenize(s))
except Exception as e:
    mlm = None
    print("Modelni yuklab boʻlmadi (Colab da internet kerak):", e)

Model soʻzlarni qism soʻzlarga (tokenlarga) boʻladi, shuning uchun soʻz vektori — uning qism soʻzlari vektorlarining oʻrtachasi. `ʻ` belgisi model oʻqitilgan matnlarda kam uchraydi, shuning uchun «koʻk», «Koʻkda», «koʻkka» turlicha boʻlinishi mumkin: qism soʻzlar roʻyxatiga qarang. Oʻsha oltita gapni olamiz va har bir qatlamda koʻramiz: bir maʼnodagi qoʻllanishlar turli maʼnodagilarga qaraganda bir-biriga qanchalik yaqinroq. 0-qatlam — kirish embeddinglari (deyarli statik: qism soʻz vektoriga faqat pozitsiya vektori qoʻshilgan).

In [ ]:
real_sentences = list(toy_sentences)

def word_vectors(sentence, prefix):
    # prefix bilan boshlanadigan soʻz vektorlari barcha qatlamlarda: (qatlamlar + 1, hidden)
    start = sentence.lower().find(prefix)
    end = start + re.search(r"\w+", sentence[start:]).end()
    enc = tok(sentence, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0].tolist()
    ids = [j for j, (a, b) in enumerate(offsets) if b > a and a < end and b > start]
    with torch.no_grad():
        hs = mlm(**enc, output_hidden_states=True).hidden_states
    return np.stack([h[0, ids].mean(0).numpy() for h in hs]), [tok.convert_ids_to_tokens(int(enc["input_ids"][0, j])) for j in ids]

if globals().get("mlm") is not None:
    try:
        vecs, senses = [], []
        for s, sense in real_sentences:
            v, pieces = word_vectors(s, TARGET.lower())
            vecs.append(v)
            senses.append(sense)
            print(f"{s:<36} qism soʻzlar: {pieces}")
        vecs = np.array(vecs)                                  # (gaplar, qatlamlar + 1, hidden)
        same = np.array([[a == b for b in senses] for a in senses])
        off = ~np.eye(len(senses), dtype=bool)
        gaps = []
        print("\nqatlam oʻziniki begona farq")
        for L in range(vecs.shape[1]):
            Vn = vecs[:, L] / np.linalg.norm(vecs[:, L], axis=1, keepdims=True)
            C = Vn @ Vn.T
            s_in, s_out = C[same & off].mean(), C[~same].mean()
            gaps.append(s_in - s_out)
            print(f"{L:5d}   {s_in:.3f}  {s_out:.3f}   {s_in - s_out:+.3f}")
        best = int(np.argmax(gaps))
        Vn = vecs[:, best] / np.linalg.norm(vecs[:, best], axis=1, keepdims=True)
        labels = [f"{i + 1}:{s}" for i, s in enumerate(senses)]
        print(f"\nkosinuslar qatlamda {best}:")
        print(pd.DataFrame(Vn @ Vn.T, index=labels, columns=labels).round(2).to_string())
        plt.figure(figsize=(6, 3.2))
        plt.plot(gaps, marker="o")
        plt.xlabel("qatlam")
        plt.ylabel("oʻziniki − begona (kosinus)")
        plt.title("«" + TARGET + "» maʼnolari qayerda farqlanadi")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("Xato:", e)

**Niqob ostidagi soʻzni top.** BERT ga oʻxshash modellar yopilgan soʻzlarni ikki tomondagi kontekst boʻyicha tiklashga oʻrgatilgan (masked language modeling). Model qaysi soʻzlarni ehtimolli deb hisoblashini koʻring. Diqqat qiling: model internet matnlarida oʻqitilgan, ularda `ʻ` oʻrniga koʻpincha `'` yoki `‘` yoziladi, bu bashoratga taʼsir qilishi mumkin (5-topshiriqqa qarang).

In [ ]:
real_masked = [
    "Oʻzbekistonning poytaxti — [MASK].",
    "Men har kuni ertalab bir piyola [MASK] ichaman.",
    "Mushuk devordan sakray olmadi, chunki [MASK] charchagan edi.",
    "Bugun havo juda [MASK] boʻldi.",
]

if globals().get("mlm") is not None:
    try:
        from transformers import pipeline
        fill = pipeline("fill-mask", model=mlm, tokenizer=tok)
        for s in real_masked:
            s = s.replace("[MASK]", tok.mask_token)
            print(s)
            for pred in fill(s, top_k=5):
                print(f"   {pred['token_str'].strip():<14} {pred['score']:.3f}")
    except Exception as e:
        print("Xato:", e)

## 6. Topshiriqlar

1. **Diqqat harorati.** 1-boʻlimda $\sqrt{d}$ ga boʻlishni olib tashlang, soʻng ballarni 2 oʻrniga 4 ga boʻling. Olmosh satridagi «charchagan» vazni qanday oʻzgaradi? Uch variantda shu satrning entropiyasini (bitlarda) hisoblang.
2. **Yangi kesim.** `LEX` ga mos belgili «dangasa» soʻzini qoʻshing va ikki qatlamli modelni «mushuk devordan sakray olmadi chunki u dangasa edi» gapida ishga tushiring. Soʻng «…chunki u juda eski edi» gapida («eski» soʻzini ham qoʻshing). Model antetsedentni toʻgʻri topadimi?
3. **Boshlar.** 3-boʻlimda `h = 4` va `h = 6` qoʻying. `Qh` va `heads` oʻlchamlarini chiqaring. Nega `d_model = 12` da `h = 5` ishlamaydi?
4. **Sababiy niqob.** Tajriba bilan isbotlang: sababiy niqob bilan `Xr` ning oxirgi beshta satri tasodifiy vektorlarga almashtirilsa, birinchi uchta token chiqishi oʻzgarmaydi. Niqobsiz-chi?
5. **(Colab)** `real_masked` dagi gaplarda `ʻ` ni `'` ga almashtiring (`s.replace("ʻ", "'")`) va bashoratlarni solishtiring. Soʻng `real_sentences` ga har bir maʼno uchun yana bittadan gap qoʻshing va «oʻziniki − begona» farqi eng katta boʻlgan qatlamni toping.

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing